In [1]:
import re

In [2]:
from declaration import VariableDeclaration, HybridProgramDeclaration, PredicateDeclaration

In [3]:
from KyxParser import KyxParser

In [4]:
kyx_path = 'D:/Reinforcement learning/kyx_parser/kyxs/CoastingCar_with_pred.kyx'
kyx_folder_path = 'D:/Reinforcement learning/kyx_parser/kyxs'
template = 'D:/Reinforcement learning/kyx_parser/env_template.py'

In [5]:
def read_file_as_string(filepath):
    """Reads the entire file into a single string.

    Args:
        filepath: The path to the text file.

    Returns:
        The file content as a string, or None if the file doesn't exist.
    """
    try:
        with open(filepath, 'r') as file:
            file_content = file.read()
            return file_content
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return None

# Example usage:
kyx_content = read_file_as_string(kyx_path)
template_content = read_file_as_string(template)

In [6]:
print(kyx_content)

ArchiveEntry "Coasting Car"
Description "Coasting cars can stop".

Definitions       /* function symbols cannot change their value */
  Real b;         /* braking capabilities */
  Real ep;        /* reaction time */

  /* precondition */
  Bool precondition(Real v, Real m, Real x) <-> speedConstraint(v, m, x) &  v >= 0 & b > 0;
  
  Bool speedConstraint(Real v, Real m, Real x) <-> v^2<=2*b*(m-x);

  /* controller */
  HP controller ::= {a:=-b; ++ ?v^2<=2*b*(m-x-ep*v); a:=0;};

  /* step */
  HP stepFunction ::= {x'=v, v'=a, t'=1 & v>=0 & t<=ep};

  /* done */
  Bool doneCondition(Real x, Real m) <-> x <= m;

End.

ProgramVariables  /* program variables may change their value over time */
  Real x;         /* position of the car */
  Real v;         /* velocity of the car */
  Real a;         /* acceleration of the car */
  Real m;         /* stop sign position */
  Real t;         /* clock variable */
End.

Problem           /* differential dynamic logic formula */
    precondition(v,

In [7]:
# Example usage:
parser = KyxParser()
parsed_data = parser.parse(kyx_content) # kyx_content is your KYX file content
print(parsed_data)

declarations:  {'b': 'Real', 'ep': 'Real', 'Bool precondition(Real v, Real m, Real x) ': '<-> speedConstraint(v, m, x) &  v >= 0 & b > 0;', 'Bool speedConstraint(Real v, Real m, Real x) ': '<-> v^2<=2*b*(m-x);', '?v^2<=2*b*(m-x-ep*v)': '++', 'Bool doneCondition(Real x, Real m) ': '<-> x <= m;'}
declarations:  {'x': 'Real', 'v': 'Real', 'a': 'Real', 'm': 'Real', 't': 'Real'}
{'Definitions': {'b': 'Real', 'ep': 'Real', 'Bool precondition(Real v, Real m, Real x) ': '<-> speedConstraint(v, m, x) &  v >= 0 & b > 0;', 'Bool speedConstraint(Real v, Real m, Real x) ': '<-> v^2<=2*b*(m-x);', '?v^2<=2*b*(m-x-ep*v)': '++', 'Bool doneCondition(Real x, Real m) ': '<-> x <= m;'}, 'ProgramVariables': {'x': 'Real', 'v': 'Real', 'a': 'Real', 'm': 'Real', 't': 'Real'}, 'Problem': {'precondition': 'precondition(v, m, x)', 'model': ['{\n        controller', 't := 0', 'stepFunction', '}'], 'invariant': 'speedConstraint(v, m, x)', 'safety_condition': 'doneCondition(x, m)'}}


In [11]:
import re

def _parse_declarations(self, declarations_string):
    """Parses declaration strings (Definitions, ProgramVariables) into a dictionary.
       Handles more complex formats including comments and function definitions.
    """
    declarations = {}
    lines = declarations_string.splitlines()
    in_definition_block = False  # Flag to track if we're inside a definition block

    for line in lines:
        line = line.strip()
        if line.startswith("Definitions"):
            in_definition_block = True
            continue
        elif line == "End.":
            in_definition_block = False
            continue
        elif not line or line.startswith("/*"):  # Ignore empty lines and comments
            continue

        if in_definition_block:
            # Use regex to find declarations within the definition block
            match = re.match(r'\s*(\w+)\s+(\w+);', line) # Matches "type name;"
            if match:
                var_type, var_name = match.groups()
                declarations[var_name] = var_type
            else:
                # Handle cases where the declaration format is not "type name;"
                print(f"Warning: Could not parse declaration: {line}")

    return declarations

# Example Usage:
declarations_string = """
Definitions       /* function symbols cannot change their value */
  Real b;         /* braking capabilities */
  Real ep;        /* reaction time */

  /* precondition */
  Bool precondition(Real v, Real m, Real x) <-> speedConstraint(v, m, x) &  v >= 0 & b > 0;

  Bool speedConstraint(Real v, Real m, Real x) <-> v^2<=2*b*(m-x);

  /* controller */
  HP controller(Real v, Real m, Real x) ::= {a:=-b; ++ ?v^2<=2*b*(m-x-ep*v); a:=0;};

  /* step */
  HP stepFunction(Real v, Real a, Real t) ::= {x'=v, v'=a, t'=1 & v>=0 & t<=ep};

  /* done */
  Bool doneCondition(Real x, Real m) <-> x <= m;

End.
"""

declarations = _parse_declarations(None, declarations_string)  # Assuming 'self' is not needed here.
print(declarations)

{'b': 'Real', 'ep': 'Real'}


In [12]:
import os

def iterate_files_top_level(directory):
    """Iterates over files in the specified directory only (not recursively)."""
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if os.path.isfile(filepath):  # Check if it's a file (not a directory)
            print(f"Processing file: {filepath}")
            # Add your file processing logic here

            content = read_file_as_string(filepath)
            parser = KyxParser()
            parsed_data = parser.parse(content) # kyx_content is your KYX file content
            print(parsed_data)


# Example usage:
iterate_files_top_level(kyx_folder_path)

Processing file: /content/drive/MyDrive/KeYmaera_parser/kyxs/gravitational-field-moon.kyx
declarations:  {'G': 'Real', 'M': 'Real', 'm': 'Real', 'E': 'Real'}
declarations:  {'x,y': 'Real', 'v,w': 'Real'}
Error: Could not parse the problem statement. Check the format.
{'Definitions': {'G': 'Real', 'M': 'Real', 'm': 'Real', 'E': 'Real'}, 'ProgramVariables': {'x,y': 'Real', 'v,w': 'Real'}, 'Problem': {'error': 'Could not parse problem statement'}}
Processing file: /content/drive/MyDrive/KeYmaera_parser/kyxs/gravitational-field-moonExc.kyx
declarations:  {'G': 'Real', 'M': 'Real', 'm': 'Real', 'E': 'Real'}
declarations:  {'x,y': 'Real', 'v,w': 'Real'}
Error: Could not parse the problem statement. Check the format.
{'Definitions': {'G': 'Real', 'M': 'Real', 'm': 'Real', 'E': 'Real'}, 'ProgramVariables': {'x,y': 'Real', 'v,w': 'Real'}, 'Problem': {'error': 'Could not parse problem statement'}}
Processing file: /content/drive/MyDrive/KeYmaera_parser/kyxs/carExc.kyx
declarations:  {'b': 'Real'

KeyboardInterrupt: 

In [ ]:
print(kyx_content)

In [ ]:
kyx_parser(kyx_content)